# Lab 07 — RNNs and LSTMs for Sequence Modeling

**NLP Applications Laboratory | KLEF | 2026**

*From sliding filters to reading one word at a time — how recurrent networks capture what comes before, and why the vanilla version quietly forgets.*

## Objective

Lab 06 introduced CNNs for text — filters that slide over word vectors capturing local n-gram patterns. The tiny CNN nailed 48 sentences perfectly and revealed its reasoning through filter inspection.

But CNNs have a real limit: **they see through small windows**. A width-3 filter looks at 3 words at a time. What happens when meaning depends on words 6 positions apart?

Consider this sentence:

> *"The pizza looked amazing but tasted awful."*

The word `amazing` (position 3) and `awful` (position 7) together flip the meaning. No single width-3 filter can see both. A CNN with wider filters (say width-5 or width-7) might catch this specific pattern, but the general problem — **remembering earlier words while processing later ones** — is not what CNNs are designed for.

**Recurrent Neural Networks (RNNs)** were designed for exactly this. Instead of sliding filters, an RNN reads text one word at a time, updating an internal "memory" (hidden state) after each word. In principle, that hidden state can carry information from word 1 to word 100.

**In practice, plain RNNs forget quickly.** The training signal — the gradient — decays exponentially back through time steps. By the time gradients reach early words, they're too weak to update anything meaningfully. This is the famous **vanishing gradient problem** (Bengio et al., 1994).

**Long Short-Term Memory (LSTM)** networks (Hochreiter & Schmidhuber, 1997) solved this by adding *gates* — small neural circuits that explicitly control what the network remembers, forgets, and outputs at each step.

By the end of this lab, you will:

1. Understand **why sequential processing matters** — through a corpus where polarity flips based on late-sentence words.
2. Build and train a **vanilla RNN** classifier, and see it struggle on inverted sentences.
3. Build and train an **LSTM** classifier on the same data, and see it succeed.
4. **Trace the hidden state** word-by-word through one sentence for both architectures — see what each model "remembers."
5. **Visualize the vanishing gradient** — plot how gradient magnitude decays across time steps in RNN vs. LSTM. This is the empirical evidence for the 1994 paper.
6. Understand

## What you will learn before we start

This lab has **eleven phases**, organized as "understand the mechanism on a small polarity-flip corpus, then compare RNN and LSTM concretely."

### Phases 1-3 — Set up the experiment

**Phase 1** — Concept: why sequential models? The polarity-flip problem that CNNs struggle with.

**Phase 2** — Build the tiny corpus: 32 sentences × 2 classes. Half straightforward, half with `but`/`however`/`then` polarity inversions.

**Phase 3** — Represent sentences as sequences of Word2Vec vectors. Same pipeline as Lab 06.

### Phases 4-7 — Vanilla RNN

**Phase 4** — Anatomy of an RNN cell. What `h_t = tanh(W_x·x_t + W_h·h_{t-1} + b)` actually computes.

**Phase 5** — Build a vanilla RNN classifier in PyTorch (`nn.RNN`).

**Phase 6** — Train it on the polarity-flip corpus. Watch it succeed on simple sentences and fail on inverted ones.

**Phase 7** — Trace the RNN's hidden state word-by-word through one polarity-flipping sentence. See what it remembers (or forgets) by the time it reaches the crucial reversal word.

### Phases 8-9 — LSTM

**Phase 8** — Anatomy of an LSTM cell. Gates: input, forget, output. Same interface as RNN, four times the parameters.

**Phase 9** — Build, train, and trace an LSTM on the same corpus. Side-by-side comparison with RNN.

### Phase 10 — The vanishing gradient plot

The empirical moment: compute and plot how gradient magnitude decays across time steps for both models. This is the concrete evidence for why LSTM was invented.

### Phase 11 — Wrap-up + post-lab assignment

Post-lab: scale the LSTM to 20 Newsgroups. Add another row to your Lab 05-06 comparison table.

### One honest note about scale

Our tiny corpus has sentences of 4-9 words. Real RNN vs LSTM differences become dramatic on sequences of hundreds of tokens (paragraphs, documents). At our small scale we'll see *some* difference in accuracy, but the **gradient plot in Phase 10 is where the difference becomes undeniable**. Set your expectations accordingly — the visualization is the pedagogical payoff, not the accuracy number.

---

## Phase 1 — Why sequential models?

### The specific problem CNNs cannot solve

In Lab 06, our CNN used filters of width 3 — each filter looked at 3 consecutive words at once. Max-pooling then kept the single strongest match per filter. This worked beautifully for classification driven by short local phrases: "love pizza", "laptop is broken", "tastes amazing".

But consider this pair of sentences:

- Sentence A: *"The pizza looked amazing."* → clearly positive
- Sentence B: *"The pizza looked amazing but tasted awful."* → clearly negative

**These sentences share the first four words verbatim.** A CNN's width-3 filter, sliding over Sentence B, will fire strongly on "looked amazing" — and max-pooling will preserve that peak activation. The filter has no way to know that "awful" appeared later.

To catch this reversal, we'd need a filter of width 7 or more — wide enough to see "amazing" and "awful" simultaneously. But that just moves the problem: what about a 20-word sentence where the reversal appears at word 18? Making filters arbitrarily wide is not a scalable solution.

### The general problem: long-range dependencies

Real language is full of dependencies that span many words:

- **Negation across clauses**: "I didn't think, given everything we discussed yesterday, that the plan was actually bad."
- **Coreference**: "The scientist who won the Nobel Prize last year for her work on gravitational waves said today that..."
- **Long-distance agreement**: "The keys, which were on the counter next to the fruit bowl and the stack of unopened mail, are missing."

CNNs handle local patterns beautifully but struggle when meaning is *distributed* across long spans.

### The RNN idea

Instead of sliding fixed-width filters, what if the network could *read* the sentence sequentially — one word at a time — and maintain an internal running summary?

That's exactly what a Recurrent Neural Network does:

1. Start with a blank memory (hidden state = zeros).
2. Read word 1 → update memory based on word 1 + blank memory.
3. Read word 2 → update memory based on word 2 + previous memory.
4. ...
5. Read word N → produce final memory as the summary of the whole sentence.
6. Feed final memory to a classifier.

In principle, the memory at word N carries information from word 1 through word N. In practice, plain RNNs forget — the "vanishing gradient problem" that we'll visualize concretely in Phase 10.

### What we'll build

Two classifiers on the same corpus:

- **Vanilla RNN**: the 1986 architecture, simple recurrent formula
- **LSTM**: the 1997 fix with gates that explicitly control memory

Then we look inside both — trace hidden states word by word, plot gradient magnitudes — and see the difference not just in accuracy but in the mechanism itself.

---

## Phase 2 — The tiny polarity-flip corpus

To *see* whether a model handles long-range dependencies, we need a corpus specifically designed to test them. Not random sentences — carefully constructed pairs where short and long versions have opposite meanings.

We use **32 sentences × 2 classes** (positive, negative). Structure:

| Type | Count per class | Example |
|---|---|---|
| Short, straightforward | 8 | *"The pizza was delicious"* (positive) |
| Long, with reversal | 8 | *"The pizza looked amazing but tasted awful"* (negative) |

**Total: 32 sentences** (16 positive, 16 negative). Balanced classes. Sentences range from 4 to 9 words.

### Why this structure

The two subsets test different things:

- **Straightforward sentences** test whether the model handles basic sentiment at all. Both RNN and LSTM should succeed here — no long-range dependencies are needed.

- **Inverted sentences** test whether the model can update its early-formed impression based on later evidence. This is where vanilla RNN should struggle and LSTM should succeed.

If both models score well on straightforward sentences and only LSTM handles the inversions, we've isolated the exact behavioral difference between the architectures. That's a clean scientific comparison.

### Why so few sentences

Two reasons:

1. **Human-inspectable size.** We want to trace individual sentences by hand in Phase 7 and Phase 9. With 32 sentences we can afford to look at each one if needed.

2. **Statistical honesty about scale.** With so few sentences, absolute accuracy numbers are noisy — swapping one misclassification changes accuracy by 3 percentage points. We're not measuring performance here; we're measuring *behavior on specific input patterns*. The gradient plot in Phase 10 is the real evidence, not the accuracy scores.

In [1]:
# 32 sentences: 16 positive, 16 negative. Half straightforward, half with polarity reversal.
polarity_corpus = {
    'positive': [
        # Straightforward positive (8)
        "The pizza was delicious",
        "This laptop runs perfectly",
        "The movie was fantastic",
        "The coffee tastes great",
        "This book is wonderful",
        "The service was excellent",
        "The phone works beautifully",
        "This meal was amazing",
        # Reversed to positive (8): starts negative-sounding, ends positive
        "The pizza looked burnt but tasted amazing",
        "The laptop seemed slow but ran perfectly",
        "The movie started boring but became fantastic",
        "The coffee smelled odd but tasted great",
        "The book seemed dull however it was wonderful",
        "The service felt rushed but was excellent",
        "The phone looked cheap but worked beautifully",
        "The meal appeared strange but was amazing",
    ],
    'negative': [
        # Straightforward negative (8)
        "The pizza was terrible",
        "This laptop runs slowly",
        "The movie was boring",
        "The coffee tastes awful",
        "This book is dreadful",
        "The service was poor",
        "The phone works badly",
        "This meal was disgusting",
        # Reversed to negative (8): starts positive-sounding, ends negative
        "The pizza looked amazing but tasted awful",
        "The laptop seemed fast but ran slowly",
        "The movie started great but became boring",
        "The coffee smelled wonderful but tasted awful",
        "The book seemed promising however it was dreadful",
        "The service felt friendly but was poor",
        "The phone looked premium but worked badly",
        "The meal appeared delicious but was disgusting",
    ],
}

# Flatten to (sentence, label_index) pairs
polarity_labels = sorted(polarity_corpus.keys())     # ['negative', 'positive']
polarity_label2idx = {lbl: i for i, lbl in enumerate(polarity_labels)}
polarity_data = [(s, polarity_label2idx[lbl]) for lbl in polarity_labels for s in polarity_corpus[lbl]]

print(f"Total sentences   : {len(polarity_data)}")
print(f"Classes ({len(polarity_labels)})       : {polarity_labels}")
print(f"Sentence lengths  : {sorted(set(len(s.split()) for s, _ in polarity_data))}")
print(f"\nStraightforward examples:")
print(f"  [positive] {polarity_data[16][0]}")   # first straightforward positive
print(f"  [negative] {polarity_data[0][0]}")    # first straightforward negative
print(f"\nInverted examples (the interesting ones):")
print(f"  [positive] {polarity_data[24][0]}")   # first inverted positive
print(f"  [negative] {polarity_data[8][0]}")    # first inverted negative

Total sentences   : 32
Classes (2)       : ['negative', 'positive']
Sentence lengths  : [4, 7, 8]

Straightforward examples:
  [positive] The pizza was delicious
  [negative] The pizza was terrible

Inverted examples (the interesting ones):
  [positive] The pizza looked burnt but tasted amazing
  [negative] The pizza looked amazing but tasted awful


### This is the pedagogical setup: same words in different positions produce opposite meanings. Only a model that reads sequentially and remembers early words can handle this correctly.

---

## Phase 3 — Represent sentences as sequences of Word2Vec vectors

Same pipeline as Lab 06: reload Word2Vec, tokenize each sentence, look up each word's 300-dim vector, pad shorter sentences with zero vectors to match the longest.

The result is a tensor of shape `(32, max_len, 300)` — 32 sentences, each padded to the maximum sentence length, each word represented by its 300-dim Word2Vec vector. This is what our RNN and LSTM will consume.

**One important difference from Lab 06:** in Lab 06 our CNN transposed the input to `(batch, embed_dim, seq_len)` because `Conv1d` expects channels first. **RNN and LSTM don't need this transpose** — they consume input in the natural order `(batch, seq_len, embed_dim)`. One less thing to remember.

In [2]:
# Reload pretrained Word2Vec (self-contained notebook policy)
import gensim.downloader as api
print("Loading Word2Vec from cache...")
w2v = api.load('word2vec-google-news-300')
print(f"Loaded. Vocab: {len(w2v.key_to_index):,} words | Dim: {w2v.vector_size}")

Loading Word2Vec from cache...
Loaded. Vocab: 3,000,000 words | Dim: 300


In [3]:
# Which words from the polarity corpus are in Word2Vec's vocabulary?
all_words = set()
for s, _ in polarity_data:
    for w in s.lower().split():
        all_words.add(w)

in_vocab = [w for w in sorted(all_words) if w in w2v]
out_vocab = [w for w in sorted(all_words) if w not in w2v]

print(f"Total unique words in polarity corpus: {len(all_words)}")
print(f"In Word2Vec vocab                    : {len(in_vocab)}")
print(f"Missing from vocab                   : {len(out_vocab)}")
if out_vocab:
    print(f"\nMissing words: {out_vocab}")

Total unique words in polarity corpus: 55
In Word2Vec vocab                    : 55
Missing from vocab                   : 0


## Convert sentences to padded tensors

In [4]:
# Convert every sentence to a (max_len × 300) matrix, padding shorter ones with zeros
import numpy as np
tokenized = [s.lower().split() for s, _ in polarity_data]
max_len = max(len(t) for t in tokenized)
print(f"Longest sentence: {max_len} words")

def sentence_to_matrix(sentence, max_len=max_len, dim=300):
    tokens = sentence.lower().split()
    matrix = np.zeros((max_len, dim), dtype=np.float32)
    for i, tok in enumerate(tokens[:max_len]):
        if tok in w2v:
            matrix[i] = w2v[tok]
    return matrix

X_pol = np.stack([sentence_to_matrix(s) for s, _ in polarity_data])
y_pol = np.array([lbl for _, lbl in polarity_data])
print(f"Input tensor X_pol shape:  {X_pol.shape}  (sentences × positions × dims)")
print(f"Label tensor y_pol shape: {y_pol.shape}")

Longest sentence: 8 words
Input tensor X_pol shape:  (32, 8, 300)  (sentences × positions × dims)
Label tensor y_pol shape: (32,)


In [5]:
# Inspect one short sentence to see padded zeros at the end
example_idx = 0  # "The pizza was terrible" — 4 words, so positions 4-8 should be padded
print(f"Sentence: {polarity_data[example_idx][0]!r} ({len(polarity_data[example_idx][0].split())} words)")
print(f"Matrix shape: {X_pol[example_idx].shape}")
print(f"Norm of each row: {[float(f'{np.linalg.norm(row):.2f}') for row in X_pol[example_idx]]}")

Sentence: 'The pizza was terrible' (4 words)
Matrix shape: (8, 300)
Norm of each row: [1.07, 3.19, 2.06, 2.54, 0.0, 0.0, 0.0, 0.0]


---

## Phase 4 — Anatomy of a vanilla RNN cell

Before we build the model, let's understand exactly what one RNN cell computes at each time step. This is the "concept before code" cell.

### The RNN update equation

At each time step `t`, the RNN takes two inputs:
- The current word vector `x_t` (300 dims from Word2Vec)
- The previous hidden state `h_{t-1}` (H dims — a hyperparameter we choose)

It produces one output:
- The new hidden state `h_t` (H dims)

The equation is:

# h_t = tanh(W_x · x_t + W_h · h_{t-1} + b)


Where:
- `W_x` is a matrix of shape `(H, 300)` — projects the current word vector to hidden space
- `W_h` is a matrix of shape `(H, H)` — projects the previous hidden state
- `b` is a bias vector of shape `(H,)`
- `tanh` squashes the result to the range (-1, +1)

**Every time step uses the same weights.** `W_x`, `W_h`, and `b` are shared across all positions. This weight-sharing is what makes RNNs "recurrent" — the same small computation is applied over and over as the network reads through the sequence.

### Reading a sentence step by step

For our sentence "The pizza looked amazing but tasted awful":

t=0: h_0 = zeros (initial state, nothing has been read)
t=1: read "The" → h_1 = tanh(W_x·v_the + W_h·h_0 + b)
t=2: read "pizza" → h_2 = tanh(W_x·v_pizza + W_h·h_1 + b)
t=3: read "looked" → h_3 = tanh(W_x·v_looked + W_h·h_2 + b)
t=4: read "amazing" → h_4 = tanh(W_x·v_amazing + W_h·h_3 + b)
t=5: read "but" → h_5 = tanh(W_x·v_but + W_h·h_4 + b)
t=6: read "tasted" → h_6 = tanh(W_x·v_tasted + W_h·h_5 + b)
t=7: read "awful" → h_7 = tanh(W_x·v_awful + W_h·h_6 + b)


**The final hidden state `h_7` is the "summary" of the whole sentence.** It's what we feed to the classifier.

### The theoretical hope, and the practical problem

**Theoretical hope**: `h_7` encodes information from every word — including the crucial "amazing" from step 4. When the classifier sees `h_7`, it should notice both the positive "amazing" and the negative "awful" and correctly predict negative.

**Practical problem**: for the classifier to learn "when you see amazing AND awful, predict negative," the gradient signal during backpropagation must reach *back* from the output to the "amazing" at step 4. This means the gradient must pass through `W_h` matrix multiplications *three times* (steps 7 → 6 → 5 → 4).

Each time the gradient passes through `W_h`, it gets multiplied by values that are typically less than 1 (because `tanh` saturates). Multiplying small numbers together → they shrink exponentially. **By the time the gradient reaches step 4, it's near zero.** The network cannot learn to associate "amazing" with the eventual classification.

This is the **vanishing gradient problem**. It's the reason vanilla RNNs struggle with long-range dependencies — not because they can't in principle represent the pattern, but because they can't *learn* it from data.

### What we'll build

An RNN classifier with hidden size H=32. Following our tradition:

Sentence: 8 words × 300 dims
↓ (RNN reads word-by-word, updates hidden state at each step)
Hidden states: 8 × 32 dims
↓ (take the final hidden state)
Final state: 32 dims
↓ (linear layer: 32 → 2 classes)
Class logits: 2 numbers

## Build the vanilla RNN classifier in PyTorch

In [7]:
# Vanilla RNN classifier: nn.RNN → take final hidden state → linear classifier
import torch
import torch.nn as nn

class SimpleRNNClassifier(nn.Module):
    def __init__(self, embed_dim=300, hidden_size=32, n_classes=2):
        super().__init__()
        # batch_first=True → input shape (batch, seq_len, embed_dim). Natural order.
        self.rnn = nn.RNN(input_size=embed_dim,
                          hidden_size=hidden_size,
                          batch_first=True,
                          nonlinearity='tanh')
        self.classifier = nn.Linear(hidden_size, n_classes)
    def forward(self, x, return_all_states=False):
        # x: (batch, seq_len, embed_dim)
        all_states, final_state = self.rnn(x)      # all_states: (batch, seq_len, H); final_state: (1, batch, H)
        final = final_state.squeeze(0)              # → (batch, H)
        logits = self.classifier(final)             # → (batch, n_classes)
        if return_all_states:
            return logits, all_states               # for tracing hidden state evolution
        return logits

In [8]:
# Create the RNN model with default settings
torch.manual_seed(42)
rnn_model = SimpleRNNClassifier(embed_dim=300, hidden_size=32, n_classes=2)
print(rnn_model)

SimpleRNNClassifier(
  (rnn): RNN(300, 32, batch_first=True)
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)


In [9]:
# Detailed architecture summary: layer, input shape, output shape, parameters
def summarize_rnn(model, input_shape=(1, 8, 300)):
    print(f"{'Layer':<28} {'Input shape':<22} {'Output shape':<22} {'Params':>10}")
    print("=" * 82)
    x = torch.zeros(input_shape)
    print(f"{'INPUT (sentence tensor)':<28} {'—':<22} {str(tuple(x.shape)):<22} {'0':>10}")

    # RNN produces (all_states, final_state)
    all_states, final_state = model.rnn(x)
    rnn_params = sum(p.numel() for p in model.rnn.parameters())
    print(f"{'RNN(300, 32) — all states':<28} {str(tuple(x.shape)):<22} {str(tuple(all_states.shape)):<22} {rnn_params:>10,}")
    print(f"{'  └─ final state only':<28} {'—':<22} {str(tuple(final_state.shape)):<22} {'(same as above)':>10}")

    final = final_state.squeeze(0)
    print(f"{'squeeze final state':<28} {str(tuple(final_state.shape)):<22} {str(tuple(final.shape)):<22} {'0':>10}")

    logits = model.classifier(final)
    clf_params = sum(p.numel() for p in model.classifier.parameters())
    print(f"{'Linear(32, 2)':<28} {str(tuple(final.shape)):<22} {str(tuple(logits.shape)):<22} {clf_params:>10,}")

    total = sum(p.numel() for p in model.parameters())
    print("=" * 82)
    print(f"{'TOTAL TRAINABLE PARAMETERS':<73} {total:>10,}")

    # Breakdown of RNN weights
    print(f"\nRNN internal weight breakdown:")
    for name, p in model.rnn.named_parameters():
        print(f"  {name:<20} shape={tuple(p.shape)}  params={p.numel():,}")

summarize_rnn(rnn_model)

Layer                        Input shape            Output shape               Params
INPUT (sentence tensor)      —                      (1, 8, 300)                     0
RNN(300, 32) — all states    (1, 8, 300)            (1, 8, 32)                 10,688
  └─ final state only        —                      (1, 1, 32)             (same as above)
squeeze final state          (1, 1, 32)             (1, 32)                         0
Linear(32, 2)                (1, 32)                (1, 2)                         66
TOTAL TRAINABLE PARAMETERS                                                    10,754

RNN internal weight breakdown:
  weight_ih_l0         shape=(32, 300)  params=9,600
  weight_hh_l0         shape=(32, 32)  params=1,024
  bias_ih_l0           shape=(32,)  params=32
  bias_hh_l0           shape=(32,)  params=32


In [10]:
# Convert numpy arrays to PyTorch tensors, set up loss and optimizer
X_pol_t = torch.tensor(X_pol, dtype=torch.float32)
y_pol_t = torch.tensor(y_pol, dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.01)
print(f"X tensor shape: {X_pol_t.shape}, dtype: {X_pol_t.dtype}")
print(f"y tensor shape: {y_pol_t.shape}, dtype: {y_pol_t.dtype}")

X tensor shape: torch.Size([32, 8, 300]), dtype: torch.float32
y tensor shape: torch.Size([32]), dtype: torch.int64


## Train the vanilla RNN

In [11]:
# Full-batch training on 32 polarity sentences
EPOCHS = 300
rnn_losses, rnn_accuracies = [], []
for epoch in range(EPOCHS):
    rnn_optimizer.zero_grad()
    logits = rnn_model(X_pol_t)
    loss = loss_fn(logits, y_pol_t)
    loss.backward()
    rnn_optimizer.step()
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == y_pol_t).float().mean().item()
    rnn_losses.append(loss.item()); rnn_accuracies.append(acc)
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:>3}  loss = {loss.item():.4f}  train_acc = {acc:.4f}")

Epoch   0  loss = 0.7178  train_acc = 0.4375
Epoch  50  loss = 0.0001  train_acc = 1.0000
Epoch 100  loss = 0.0000  train_acc = 1.0000
Epoch 150  loss = 0.0000  train_acc = 1.0000
Epoch 200  loss = 0.0000  train_acc = 1.0000
Epoch 250  loss = 0.0000  train_acc = 1.0000
Epoch 299  loss = 0.0000  train_acc = 1.0000


# Check per-sentence predictions and separate straightforward vs inverted accuracy

In [12]:
# Break down performance: does the RNN get straightforward AND inverted sentences right?
rnn_model.eval()
with torch.no_grad():
    all_preds = rnn_model(X_pol_t).argmax(dim=1).numpy()

# In our polarity_data, indices 0-7 are straightforward negative, 8-15 are inverted negative,
# indices 16-23 are straightforward positive, 24-31 are inverted positive
groups = [
    ('Straightforward negative', range(0, 8)),
    ('Inverted negative       ', range(8, 16)),
    ('Straightforward positive', range(16, 24)),
    ('Inverted positive       ', range(24, 32)),
]

print(f"{'Group':<28} {'Correct':<10} {'Accuracy'}")
print("-" * 50)
for name, indices in groups:
    correct = sum(1 for i in indices if all_preds[i] == y_pol[i])
    total = len(indices)
    print(f"{name:<28} {correct}/{total:<8} {correct/total:.3f}")

print("\nSentence-by-sentence:")
for name, indices in groups:
    print(f"\n{name}:")
    for i in indices:
        pred_label = polarity_labels[all_preds[i]]
        true_label = polarity_labels[y_pol[i]]
        mark = "✓" if all_preds[i] == y_pol[i] else "✗"
        print(f"  {mark} pred={pred_label:<9} true={true_label:<9} — {polarity_data[i][0]}")

Group                        Correct    Accuracy
--------------------------------------------------
Straightforward negative     8/8        1.000
Inverted negative            8/8        1.000
Straightforward positive     8/8        1.000
Inverted positive            8/8        1.000

Sentence-by-sentence:

Straightforward negative:
  ✓ pred=negative  true=negative  — The pizza was terrible
  ✓ pred=negative  true=negative  — This laptop runs slowly
  ✓ pred=negative  true=negative  — The movie was boring
  ✓ pred=negative  true=negative  — The coffee tastes awful
  ✓ pred=negative  true=negative  — This book is dreadful
  ✓ pred=negative  true=negative  — The service was poor
  ✓ pred=negative  true=negative  — The phone works badly
  ✓ pred=negative  true=negative  — This meal was disgusting

Inverted negative       :
  ✓ pred=negative  true=negative  — The pizza looked amazing but tasted awful
  ✓ pred=negative  true=negative  — The laptop seemed fast but ran slowly
  ✓ pred=negative

---

## Phase 7 — What does the RNN "remember"? Tracing hidden state word-by-word

The RNN classified every sentence correctly, including the inverted ones. That's *what* it did. Now we look at *how* — by tracing the hidden state as the network reads one specific inverted sentence, word by word.

### What we'll look at

For the sentence *"The pizza looked amazing but tasted awful"*, we'll:

1. Feed it to the RNN and get all 8 hidden states (one per time step).
2. Show the **norm** of the hidden state at each step. A large norm means "the hidden state is far from the origin — the network is actively representing something." A small norm means "hidden state is near zero — little information is stored."
3. Show the **change** between consecutive hidden states: `||h_t - h_{t-1}||`. Large changes indicate the network updated its representation significantly at that word. Small changes indicate the current word barely affected the running summary.

### What to look for

**If the network is genuinely tracking word-by-word meaning**, we should see:

- Small change at "The" (a stopword — should not shift the state much)
- Meaningful updates at content words: "pizza", "looked", "amazing", "but", "tasted", "awful"
- **Especially large updates at the *reversal words*** — "but" and "awful". These are the moments where the model has to substantially revise its impression.

If we see zero change at the reversal words, the network isn't tracking meaning — it's just relying on the presence of "awful" alone (a bag-of-words shortcut that happens to work).

### What this doesn't tell us

We can't directly see "what the hidden state means" — it's just 32 numbers per time step. We can only measure how *actively* those numbers change. Interpreting the semantics of individual dimensions of a learned RNN hidden state is genuinely hard, and researchers have written papers on this.

For our teaching purposes, the norm-and-change view is enough: it shows *whether* the network is treating each word as informative.

## Trace the RNN hidden state through one sentence

In [13]:
# Trace hidden state evolution for one inverted-negative sentence
trace_idx = 8   # "The pizza looked amazing but tasted awful"
sentence, true_label = polarity_data[trace_idx]
print(f"Tracing: '{sentence}'  (true class: {polarity_labels[true_label]})\n")

rnn_model.eval()
with torch.no_grad():
    x = X_pol_t[trace_idx:trace_idx+1]                      # (1, 8, 300)
    logits, all_states = rnn_model(x, return_all_states=True)
    hidden_states = all_states[0]                            # (8, 32) — drop batch dim
    predicted = logits.argmax(dim=1).item()

# Include an implicit h_0 = zeros for computing the change at t=1
h_prev = torch.zeros(hidden_states.shape[1])
tokens = sentence.lower().split() + ['<pad>'] * (hidden_states.shape[0] - len(sentence.split()))

print(f"{'t':<4} {'word':<12} {'||h_t||':>10} {'||h_t - h_{t-1}||':>18}")
print("-" * 48)
for t in range(hidden_states.shape[0]):
    h_t = hidden_states[t]
    norm = h_t.norm().item()
    change = (h_t - h_prev).norm().item()
    print(f"{t+1:<4} {tokens[t]:<12} {norm:>10.4f} {change:>18.4f}")
    h_prev = h_t

print(f"\nFinal prediction: {polarity_labels[predicted]}  {'✓' if predicted == true_label else '✗'}")

Tracing: 'The pizza looked amazing but tasted awful'  (true class: negative)

t    word            ||h_t||  ||h_t - h_{t-1}||
------------------------------------------------
1    the              1.2759             1.2759
2    pizza            3.2868             3.4974
3    looked           2.9664             4.5825
4    amazing          4.7228             6.5319
5    but              5.2175             7.5419
6    tasted           4.2204             8.2604
7    awful            4.5091             7.1410
8    <pad>            5.1790             8.5344

Final prediction: negative  ✓


How does the hidden state norm change across time steps? If it grows monotonically, the network is accumulating information. If it stays roughly constant, the network's memory has bounded capacity (which is one interpretation of "forgetting").
Which words produce the biggest updates (||h_t - h_{t-1}||)? In a well-behaved network:
Stopword the → small change
Content words pizza, looked, amazing, awful → large changes
Reversal word but → hopefully large change (this word carries "revise your impression")
The pad token at t=8 — since we appended <pad> and the padded position has a zero input vector, the hidden state change at t=8 should be smaller than at meaningful words. This is a signature that the network correctly identifies uninformative positions.

# Phase 8 intro: the LSTM

---

## Phase 8 — LSTM: fixing the vanishing gradient

Vanilla RNN worked on our 8-word sentences — but the underlying architectural weakness (vanishing gradients) is real and would surface on longer sequences (paragraphs, documents). In 1997, Hochreiter & Schmidhuber published **Long Short-Term Memory (LSTM)** — a modification of the RNN that adds explicit *gates* to control what the network remembers, forgets, and outputs at each step.

### The intuition

A vanilla RNN's hidden state is a single vector that gets *entirely rewritten* at each time step:

h_t = tanh(W_x·x_t + W_h·h_{t-1} + b)


If the current word `x_t` overwhelms the equation, previous state `h_{t-1}` gets erased. Gradients trying to flow back through many such rewrites get suppressed exponentially.

An LSTM instead maintains **two** state vectors:

- **Cell state** `c_t` — the "long-term memory," updated additively (not multiplicatively) so old information can persist
- **Hidden state** `h_t` — the "current output," derived from the cell state through a controlled gate

Three learned gates decide what happens at each step:

- **Forget gate** `f_t` — how much of the previous cell state to keep (values in [0, 1])
- **Input gate** `i_t` — how much of a candidate new memory to add (values in [0, 1])
- **Output gate** `o_t` — how much of the cell state to expose as this step's hidden state (values in [0, 1])

The cell state update is:

c_t = f_t ⊙ c_{t-1} + i_t ⊙ c̃_t


Where `⊙` is element-wise multiplication and `c̃_t` is a candidate new memory computed from the current input. **Crucially**, if the forget gate outputs 1 and the input gate outputs 0, then `c_t = c_{t-1}` — the previous memory passes through *unchanged*. Gradient flow along this identity path doesn't vanish.

### Why this matters

Vanilla RNN: at every time step, the hidden state is rewritten through `tanh`, which saturates. Gradient signals shrink by a factor <1 at each step. After 100 steps, gradients are essentially zero.

LSTM: cell state updates via addition, not through `tanh` at the recurrent path. If the network learns to set `f_t ≈ 1` for information worth preserving, that information (and its gradient) flows through many time steps intact.

**LSTM's gates are learned end-to-end** — the network figures out from data which gate values solve the task. We don't program the gates; we just train.

### Parameter cost

An LSTM has 4 sets of weights (one for each of: input gate, forget gate, output gate, candidate memory) where a vanilla RNN has 1 set. So LSTM has roughly **4× the parameters** of an equivalent-size RNN.

For our hidden size of 32:
- Vanilla RNN: ~10,656 recurrent parameters
- LSTM: ~42,624 recurrent parameters

Small in absolute terms, but a real 4× multiplier.

### What we'll do

Build an LSTM with the **exact same interface** as our RNN — same input/output shapes, same training loop, same evaluation. In PyTorch, this is a one-line change: swap `nn.RNN` for `nn.LSTM`. Then we compare:

1. Training curves (does LSTM converge faster or to lower loss?)
2. Per-group accuracy (same 100%? or better on inversions?)
3. Hidden state trace (does LSTM's hidden state behave differently on the reversal word?)

Phase 10 is where the real difference becomes undeniable — the gradient norm plot showing RNN's decay vs LSTM's stability across time steps.

# Consolidated LSTM cell: define, train, evaluate, trace

In [15]:
# ============================================================
# CONSOLIDATED: LSTM — train, evaluate, trace one sentence
# ============================================================

# ---------- Model definition ----------
class SimpleLSTMClassifier(nn.Module):
    def __init__(self, embed_dim=300, hidden_size=32, n_classes=2):
        super().__init__()
        # nn.LSTM: same interface as nn.RNN, four internal gates
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, n_classes)
    def forward(self, x, return_all_states=False):
        # LSTM returns (all_states, (final_hidden, final_cell))
        all_states, (final_h, final_c) = self.lstm(x)
        logits = self.classifier(final_h.squeeze(0))
        if return_all_states:
            return logits, all_states
        return logits

# ---------- Training setup ----------
torch.manual_seed(42)
lstm_model = SimpleLSTMClassifier(embed_dim=300, hidden_size=32, n_classes=2)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01)

total_params = sum(p.numel() for p in lstm_model.parameters())
print(f"LSTM total trainable parameters: {total_params:,}  (RNN was 10,722)")

# ---------- Training loop ----------
EPOCHS = 300
lstm_losses, lstm_accuracies = [], []
print("\nTraining LSTM...")
for epoch in range(EPOCHS):
    lstm_optimizer.zero_grad()
    logits = lstm_model(X_pol_t)
    loss = loss_fn(logits, y_pol_t)
    loss.backward()
    lstm_optimizer.step()
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == y_pol_t).float().mean().item()
    lstm_losses.append(loss.item()); lstm_accuracies.append(acc)
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"  Epoch {epoch:>3}  loss={loss.item():.4f}  acc={acc:.4f}")

# ---------- Per-group evaluation ----------
lstm_model.eval()
with torch.no_grad():
    all_preds = lstm_model(X_pol_t).argmax(dim=1).numpy()
print(f"\n{'Group':<28} {'Accuracy'}")
print("-" * 40)
for name, indices in groups:
    correct = sum(1 for i in indices if all_preds[i] == y_pol[i])
    print(f"{name:<28} {correct}/{len(indices)} = {correct/len(indices):.3f}")

# ---------- Trace same sentence as RNN ----------
trace_idx = 8   # "The pizza looked amazing but tasted awful"
sentence, true_label = polarity_data[trace_idx]
print(f"\nTracing hidden state: '{sentence}'  (true: {polarity_labels[true_label]})")
with torch.no_grad():
    x = X_pol_t[trace_idx:trace_idx+1]
    logits, all_states = lstm_model(x, return_all_states=True)
    hidden_states = all_states[0]
    predicted = logits.argmax(dim=1).item()

h_prev = torch.zeros(hidden_states.shape[1])
tokens = sentence.lower().split() + ['<pad>'] * (hidden_states.shape[0] - len(sentence.split()))
print(f"\n{'t':<4} {'word':<12} {'||h_t||':>10} {'||h_t-h_{t-1}||':>18}")
print("-" * 48)
for t in range(hidden_states.shape[0]):
    h_t = hidden_states[t]
    print(f"{t+1:<4} {tokens[t]:<12} {h_t.norm().item():>10.4f} {(h_t - h_prev).norm().item():>18.4f}")
    h_prev = h_t
print(f"\nLSTM prediction: {polarity_labels[predicted]}  {'✓' if predicted == true_label else '✗'}")

LSTM total trainable parameters: 42,818  (RNN was 10,722)

Training LSTM...
  Epoch   0  loss=0.7058  acc=0.5000
  Epoch  50  loss=0.0000  acc=1.0000
  Epoch 100  loss=0.0000  acc=1.0000
  Epoch 150  loss=0.0000  acc=1.0000
  Epoch 200  loss=0.0000  acc=1.0000
  Epoch 250  loss=0.0000  acc=1.0000
  Epoch 299  loss=0.0000  acc=1.0000

Group                        Accuracy
----------------------------------------
Straightforward negative     8/8 = 1.000
Inverted negative            8/8 = 1.000
Straightforward positive     8/8 = 1.000
Inverted positive            8/8 = 1.000

Tracing hidden state: 'The pizza looked amazing but tasted awful'  (true: negative)

t    word            ||h_t||    ||h_t-h_{t-1}||
------------------------------------------------
1    the              0.7490             0.7490
2    pizza            1.7596             1.1534
3    looked           2.3159             1.0061
4    amazing          3.4912             3.8555
5    but              3.0767             1.685

Step	Word	Update magnitude
1	the	0.7490 (baseline — first word from zero state)
2	pizza	1.1534 (moderate — content word)
3	looked	1.0061 (moderate — content word)
4	amazing	3.8555 ← HUGE spike
5	but	1.6857 (moderate)
6	tasted	2.8123 (large)
7	awful	3.7392 ← another HUGE spike
8	<pad>	1.6132 (still nonzero — network processing pad token)

# Direct side-by-side comparison

In [16]:
# Side-by-side comparison of RNN and LSTM hidden state evolution on the same sentence
rnn_model.eval(); lstm_model.eval()

trace_idx = 8
sentence, true_label = polarity_data[trace_idx]
print(f"Tracing both models on: '{sentence}'")
print(f"True class: {polarity_labels[true_label]}\n")

with torch.no_grad():
    x = X_pol_t[trace_idx:trace_idx+1]
    _, rnn_states = rnn_model(x, return_all_states=True)
    _, lstm_states = lstm_model(x, return_all_states=True)
    rnn_h = rnn_states[0]     # (8, 32)
    lstm_h = lstm_states[0]   # (8, 32)

tokens = sentence.lower().split() + ['<pad>'] * (rnn_h.shape[0] - len(sentence.split()))

print(f"{'t':<3} {'word':<10} {'RNN ||h||':>12} {'LSTM ||h||':>12} {'RNN Δh':>10} {'LSTM Δh':>10}")
print("-" * 66)
rnn_prev = torch.zeros(rnn_h.shape[1]); lstm_prev = torch.zeros(lstm_h.shape[1])
for t in range(rnn_h.shape[0]):
    rnn_change = (rnn_h[t] - rnn_prev).norm().item()
    lstm_change = (lstm_h[t] - lstm_prev).norm().item()
    print(f"{t+1:<3} {tokens[t]:<10} {rnn_h[t].norm().item():>12.4f} {lstm_h[t].norm().item():>12.4f} "
          f"{rnn_change:>10.4f} {lstm_change:>10.4f}")
    rnn_prev = rnn_h[t]; lstm_prev = lstm_h[t]

Tracing both models on: 'The pizza looked amazing but tasted awful'
True class: negative

t   word          RNN ||h||   LSTM ||h||     RNN Δh    LSTM Δh
------------------------------------------------------------------
1   the              1.2759       0.7490     1.2759     0.7490
2   pizza            3.2868       1.7596     3.4974     1.1534
3   looked           2.9664       2.3159     4.5825     1.0061
4   amazing          4.7228       3.4912     6.5319     3.8555
5   but              5.2175       3.0767     7.5419     1.6857
6   tasted           4.2204       2.2487     8.2604     2.8123
7   awful            4.5091       4.0286     7.1410     3.7392
8   <pad>            5.1790       4.1347     8.5344     1.6132
